# Quantization and Distillation — Exercises

[← Mixture of Experts and Routing](../10-Mixture-of-Experts-and-Routing/notes.md) | [Home](../../README.md) | [RAG Math and Retrieval →](../12-RAG-Math-and-Retrieval/notes.md)

8 exercises covering quantization fundamentals, GPTQ compensation, distillation loss,
temperature effects, QLoRA memory, sensitivity analysis, and capacity gap design.

In [ ]:
# === Setup ===
import numpy as np
np.random.seed(42)

try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

def print_table(headers, rows, col_width=14):
    hdr = ' | '.join(h.center(col_width) for h in headers)
    print(hdr)
    print('-' * len(hdr))
    for row in rows:
        print(' | '.join(str(v).center(col_width) for v in row))

def fmt_sci(x, p=4):
    return f'{x:.{p}e}'

def fmt_vec(v, p=4):
    return '[' + ', '.join(f'{x:.{p}f}' for x in v) + ']'

def fmt_bytes(n_bytes):
    if n_bytes >= 1e9:
        return f'{n_bytes/1e9:.2f} GB'
    elif n_bytes >= 1e6:
        return f'{n_bytes/1e6:.2f} MB'
    return f'{n_bytes/1e3:.2f} KB'

def softmax(x, temperature=1.0):
    x = np.asarray(x, dtype=np.float64)
    z = x / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()

def kl_divergence(p, q):
    p = np.asarray(p, dtype=np.float64)
    q = np.asarray(q, dtype=np.float64)
    mask = p > 1e-15
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

def entropy(p):
    p = np.asarray(p, dtype=np.float64)
    mask = p > 1e-15
    return -np.sum(p[mask] * np.log(p[mask]))

print('Setup complete ✓')

## Exercise 1: Uniform Quantization by Hand

Given weights `w = [-1.2, 0.3, 0.8, -0.5, 1.5, 0.1]` and `b = 3` bits:

1. Compute scale `s` for symmetric quantization (range `[-4, 3]` for 3-bit signed)
2. Compute zero point `z`
3. Quantize each weight: `x_q = clamp(round(x/s + z), -4, 3)`
4. Dequantize: `x_hat = s * (x_q - z)`
5. Compute MSE between original and dequantized
6. Compare with theoretical MSE = s²/12

In [ ]:
# === Exercise 1: Scaffold ===

w = np.array([-1.2, 0.3, 0.8, -0.5, 1.5, 0.1])
bits = 3

# TODO: Step 1 — Compute symmetric quantization range
q_max = None  # 2^(b-1) - 1
q_min = None  # -2^(b-1)

# TODO: Step 2 — Compute scale s = max(|w|) / q_max
s = None
z = None  # Symmetric → z = 0

# TODO: Step 3 — Quantize each weight
x_q = None  # clamp(round(w/s + z), q_min, q_max)

# TODO: Step 4 — Dequantize
x_hat = None  # s * (x_q - z)

# TODO: Step 5 — Compute MSE
mse = None

# TODO: Step 6 — Theoretical MSE
theoretical_mse = None  # s²/12

# Print results
# print(f'Scale s = {s}')
# print(f'Quantized: {x_q}')
# print(f'Dequantized: {x_hat}')
# print(f'Errors: {w - x_hat}')
# print(f'MSE = {mse}, Theoretical = {theoretical_mse}')

In [ ]:
# === Exercise 1: Solution ===

w = np.array([-1.2, 0.3, 0.8, -0.5, 1.5, 0.1])
bits = 3

# Step 1: 3-bit signed integer range
q_max = 2**(bits-1) - 1   # 3
q_min = -2**(bits-1)       # -4
print(f'Quantization range: [{q_min}, {q_max}] ({2**bits} levels)')

# Step 2: Scale (symmetric)
x_absmax = np.max(np.abs(w))  # 1.5
s = x_absmax / q_max          # 1.5 / 3 = 0.5
z = 0  # Symmetric
print(f'x_absmax = {x_absmax}')
print(f'Scale s = {x_absmax} / {q_max} = {s}')
print(f'Zero point z = {z}')

# Step 3: Quantize
x_q = np.clip(np.round(w / s + z), q_min, q_max).astype(int)
print(f'\nQuantization: x_q = clamp(round(w/{s} + {z}), {q_min}, {q_max})')
for i, (wi, qi) in enumerate(zip(w, x_q)):
    raw = wi / s + z
    rounded = round(raw)
    clamped = max(q_min, min(q_max, rounded))
    print(f'  w[{i}] = {wi:+.1f} → {raw:+.2f} → round={rounded:+d} → clamp={clamped:+d}')

# Step 4: Dequantize
x_hat = s * (x_q - z)
print(f'\nDequantized: {fmt_vec(x_hat, 2)}')

# Step 5: Error analysis
errors = w - x_hat
mse = np.mean(errors**2)
max_err = np.max(np.abs(errors))
print(f'\nErrors: {fmt_vec(errors, 4)}')
print(f'MSE = {mse:.6f}')
print(f'Max error = {max_err:.4f} (bounded by s/2 = {s/2:.4f})')

# Step 6: Theoretical comparison
theoretical_mse = s**2 / 12
print(f'\nTheoretical MSE (s²/12) = {s}² / 12 = {theoretical_mse:.6f}')
print(f'Ratio actual/theory = {mse/theoretical_mse:.2f}')
print('(Ratio differs from 1.0 because s²/12 assumes uniform distribution over quantization interval)')

## Exercise 2: NF4 vs INT4

For weights ~ N(0, 0.02²):

1. Compute optimal INT4 uniform quantization levels for range [-3σ, 3σ]
2. Look up NF4 levels (quantiles of standard normal, scaled by σ)
3. Generate 100K Gaussian weights and quantize with both
4. Compare MSE — which is better for Gaussian data and why?

In [ ]:
# === Exercise 2: Scaffold ===

sigma = 0.02
np.random.seed(42)
weights = np.random.randn(100000) * sigma

# TODO: INT4 uniform levels
# 16 levels uniformly spaced in [-3σ, 3σ]
int4_levels = None

# TODO: NF4 levels (quantiles of N(0,1) scaled by σ)
# NF4 places levels at quantile boundaries of standard normal
nf4_levels_unit = None  # 16 quantile-based levels of N(0,1)
nf4_levels = None       # Scale by σ

# TODO: Quantize each weight to nearest level
def quantize_to_levels(w, levels):
    """Assign each weight to nearest level."""
    pass

# TODO: Compute MSE for both
# mse_int4 = ...
# mse_nf4 = ...

In [ ]:
# === Exercise 2: Solution ===

from scipy.stats import norm  # For quantile computation

sigma = 0.02
np.random.seed(42)
weights = np.random.randn(100000) * sigma

# --- INT4: 16 uniform levels in [-3σ, 3σ] ---
clip_range = 3 * sigma
int4_levels = np.linspace(-clip_range, clip_range, 16)
print('INT4 uniform levels (scaled by σ/σ):')
print(f'  {[f"{l/sigma:.3f}σ" for l in int4_levels]}')

# --- NF4: quantile-based levels ---
# 16 levels: compute midpoints of 16 equal-probability bins of N(0,1)
# Bin edges at quantiles: 0/16, 1/16, ..., 16/16
bin_edges = np.array([norm.ppf((i + 0.5) / 16) for i in range(16)])
# Standard NF4 levels (approximate, from QLoRA paper)
nf4_levels_unit = np.array([
    -1.0, -0.6962, -0.5251, -0.3949, -0.2840, -0.1848, -0.0922, 0.0,
     0.0796, 0.1609, 0.2461, 0.3379, 0.4407, 0.5626, 0.7229, 1.0
])
nf4_levels = nf4_levels_unit * sigma  # Scale to match weight distribution

# Actually for proper NF4, normalise weights by absmax first
w_absmax = np.max(np.abs(weights))
w_norm = weights / w_absmax  # Normalise to [-1, 1]

print(f'\nNF4 levels (unit):')
print(f'  {[f"{l:.4f}" for l in nf4_levels_unit]}')

# --- Quantize to nearest level ---
def quantize_to_levels(w, levels):
    """Assign each weight to nearest level."""
    # For each weight, find index of nearest level
    dists = np.abs(w[:, np.newaxis] - levels[np.newaxis, :])  # (N, K)
    indices = np.argmin(dists, axis=1)
    return levels[indices]

# INT4 quantization
w_int4 = quantize_to_levels(weights, int4_levels)
mse_int4 = np.mean((weights - w_int4)**2)

# NF4 quantization (normalise → quantize → denormalise)
w_nf4_norm = quantize_to_levels(w_norm, nf4_levels_unit)
w_nf4 = w_nf4_norm * w_absmax
mse_nf4 = np.mean((weights - w_nf4)**2)

print(f'\n=== MSE Comparison ===')
print(f'INT4 uniform MSE: {fmt_sci(mse_int4)}')
print(f'NF4 quantile MSE: {fmt_sci(mse_nf4)}')
print(f'NF4 improvement:  {mse_int4/mse_nf4:.2f}x better')

print(f'\nWhy NF4 wins for Gaussian data:')
print(f'  - Most weights cluster near 0 (Gaussian peak)')
print(f'  - NF4 places more levels near 0 where density is highest')
print(f'  - INT4 wastes levels in tails where few weights exist')
print(f'  - NF4 is information-theoretically optimal for Gaussian data')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(weights, bins=200, density=True, alpha=0.5, label='Weights')
    for l in int4_levels:
        axes[0].axvline(l, color='red', alpha=0.3, linewidth=0.8)
    axes[0].set_title('INT4 Uniform Levels')
    axes[0].set_xlabel('Weight value')
    axes[0].legend()
    
    axes[1].hist(weights, bins=200, density=True, alpha=0.5, label='Weights')
    for l in nf4_levels:
        axes[1].axvline(l, color='green', alpha=0.3, linewidth=0.8)
    axes[1].set_title('NF4 Quantile Levels')
    axes[1].set_xlabel('Weight value')
    axes[1].legend()
    
    plt.tight_layout()
    plt.savefig('nf4_vs_int4.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved nf4_vs_int4.png')

## Exercise 3: GPTQ Compensation

Weight row `w = [1.2, -0.8, 0.4]`, scale `s = 0.2` (INT4 quantization).

Given Hessian `H = [[2, 1, 0.5], [1, 2, 1], [0.5, 1, 2]]`:

1. Quantize w₁ = 1.2 to nearest INT4 level (multiples of s = 0.2)
2. Compute quantization error δ₁ = w_q₁ - w₁
3. Compute H⁻¹
4. Update remaining weights: `δw = -(δ₁ / H⁻¹[0,0]) * H⁻¹[0, 1:]`
5. Compare output error with and without compensation

In [ ]:
# === Exercise 3: Scaffold ===

w = np.array([1.2, -0.8, 0.4])
s = 0.2  # Scale for INT4
H = np.array([[2, 1, 0.5],
              [1, 2, 1],
              [0.5, 1, 2]], dtype=float)

# TODO: Step 1 — Quantize w[0]
w_q0 = None  # round(1.2 / 0.2) * 0.2

# TODO: Step 2 — Compute error
delta_0 = None  # w_q0 - w[0]

# TODO: Step 3 — Compute H⁻¹
H_inv = None

# TODO: Step 4 — Update remaining weights
# update = -(delta_0 / H_inv[0,0]) * H_inv[0, 1:]

# TODO: Step 5 — Compare output error

In [ ]:
# === Exercise 3: Solution ===

w = np.array([1.2, -0.8, 0.4])
s = 0.2
H = np.array([[2.0, 1.0, 0.5],
              [1.0, 2.0, 1.0],
              [0.5, 1.0, 2.0]])

print('=== GPTQ Weight Compensation ===')
print(f'Original weights: {list(w)}')
print(f'Scale s = {s}\n')

# Step 1: Quantize w[0]
w_q0 = np.round(w[0] / s) * s
print(f'Step 1: Quantize w[0] = {w[0]}')
print(f'  round({w[0]} / {s}) = round({w[0]/s}) = {np.round(w[0]/s):.0f}')
print(f'  w_q[0] = {np.round(w[0]/s):.0f} × {s} = {w_q0}')

# Step 2: Error
delta_0 = w_q0 - w[0]
print(f'\nStep 2: Error δ₁ = {w_q0} - {w[0]} = {delta_0}')

# Step 3: H⁻¹
H_inv = np.linalg.inv(H)
print(f'\nStep 3: H⁻¹ =')
for row in H_inv:
    print(f'  {fmt_vec(row, 4)}')

# Step 4: Weight update
update = -(delta_0 / H_inv[0, 0]) * H_inv[0, 1:]
print(f'\nStep 4: Weight compensation')
print(f'  update = -({delta_0:.4f} / {H_inv[0,0]:.4f}) × {fmt_vec(H_inv[0, 1:])}')
print(f'  update = {fmt_vec(update)}')

w_compensated = w.copy()
w_compensated[0] = w_q0
w_compensated[1:] += update
print(f'\n  Original w:     {fmt_vec(w)}')
print(f'  After update:   {fmt_vec(w_compensated)}')

# Now quantize remaining weights
w_gptq = w_compensated.copy()
for j in range(1, 3):
    w_gptq[j] = np.round(w_compensated[j] / s) * s
print(f'  Fully quantized: {fmt_vec(w_gptq)}')

# Naive quantization (no compensation)
w_naive = np.round(w / s) * s
print(f'  Naive quantized: {fmt_vec(w_naive)}')

# Step 5: Compare output error
np.random.seed(42)
x_test = np.random.randn(3, 500)
out_orig = w @ x_test
out_naive = w_naive @ x_test
out_gptq = w_gptq @ x_test

mse_naive = np.mean((out_orig - out_naive)**2)
mse_gptq = np.mean((out_orig - out_gptq)**2)

print(f'\nStep 5: Output reconstruction error')
print(f'  Naive MSE: {fmt_sci(mse_naive)}')
print(f'  GPTQ MSE:  {fmt_sci(mse_gptq)}')
print(f'  Improvement: {mse_naive/mse_gptq:.2f}x')

## Exercise 4: Distillation Loss Computation

Teacher logits `z_T = [3.0, 1.0, -0.5]`, student logits `z_S = [2.5, 0.8, 0.0]`, temperature `τ = 2`.

1. Compute teacher soft distribution `P_T = softmax(z_T / τ)`
2. Compute student soft distribution `P_S = softmax(z_S / τ)`
3. Compute KL divergence `D_KL(P_T || P_S)`
4. Compute `τ²`-scaled distillation loss
5. Compute hard cross-entropy loss (true label = class 0)
6. Compute total Hinton loss with `α = 0.5`

In [ ]:
# === Exercise 4: Scaffold ===

z_T = np.array([3.0, 1.0, -0.5])
z_S = np.array([2.5, 0.8, 0.0])
tau = 2.0
true_label = 0
alpha = 0.5

# TODO: Step 1 — Teacher soft distribution
p_T = None  # softmax(z_T / tau)

# TODO: Step 2 — Student soft distribution
p_S = None

# TODO: Step 3 — KL divergence D_KL(P_T || P_S)
kl = None  # sum(p_T * log(p_T / p_S))

# TODO: Step 4 — τ²-scaled loss
distil_loss = None  # tau^2 * kl

# TODO: Step 5 — Hard CE loss
p_S_hard = None  # softmax(z_S, temperature=1)
ce_loss = None    # -log(p_S_hard[true_label])

# TODO: Step 6 — Total Hinton loss
total = None  # alpha * ce_loss + (1-alpha) * distil_loss

In [ ]:
# === Exercise 4: Solution ===

z_T = np.array([3.0, 1.0, -0.5])
z_S = np.array([2.5, 0.8, 0.0])
tau = 2.0
true_label = 0
alpha = 0.5

print('=== Distillation Loss — Step by Step ===')
print(f'Teacher logits z_T = {list(z_T)}')
print(f'Student logits z_S = {list(z_S)}')
print(f'Temperature τ = {tau}, α = {alpha}\n')

# Step 1: Teacher soft distribution
p_T = softmax(z_T, tau)
print(f'Step 1: P_T = softmax(z_T / {tau})')
print(f'  z_T/τ = {list(z_T / tau)}')
print(f'  P_T   = {fmt_vec(p_T)}\n')

# Step 2: Student soft distribution
p_S = softmax(z_S, tau)
print(f'Step 2: P_S = softmax(z_S / {tau})')
print(f'  z_S/τ = {list(z_S / tau)}')
print(f'  P_S   = {fmt_vec(p_S)}\n')

# Step 3: KL divergence
kl = kl_divergence(p_T, p_S)
print(f'Step 3: KL(P_T || P_S)')
for i in range(len(p_T)):
    term = p_T[i] * np.log(p_T[i] / p_S[i])
    print(f'  Class {i}: {p_T[i]:.4f} × ln({p_T[i]:.4f}/{p_S[i]:.4f}) = {term:.6f}')
print(f'  KL = {kl:.6f}\n')

# Step 4: τ²-scaled loss
distil_loss = tau**2 * kl
print(f'Step 4: τ² × KL = {tau}² × {kl:.6f} = {distil_loss:.6f}')
print(f'  (τ² scaling compensates for softmax gradient shrinkage at high τ)\n')

# Step 5: Hard CE loss
p_S_hard = softmax(z_S, 1.0)
ce_loss = -np.log(p_S_hard[true_label])
print(f'Step 5: Hard cross-entropy')
print(f'  P_S(τ=1) = {fmt_vec(p_S_hard)}')
print(f'  CE = -ln({p_S_hard[true_label]:.4f}) = {ce_loss:.6f}\n')

# Step 6: Total Hinton loss
total = alpha * ce_loss + (1 - alpha) * distil_loss
print(f'Step 6: Total = α·CE + (1-α)·τ²·KL')
print(f'  = {alpha}×{ce_loss:.6f} + {1-alpha}×{distil_loss:.6f}')
print(f'  = {alpha*ce_loss:.6f} + {(1-alpha)*distil_loss:.6f}')
print(f'  = {total:.6f}')

## Exercise 5: Temperature Effect on Dark Knowledge

Given logits `[4.0, 2.0, 0.5, -1.0]`:

1. Compute softmax at τ = 0.5, 1.0, 2.0, 5.0
2. Compute Shannon entropy at each temperature
3. Compute effective number of classes = exp(H)
4. Analyse when "dark knowledge" becomes visible
5. What information does τ = 5 reveal that τ = 1 hides?

In [ ]:
# === Exercise 5: Scaffold ===

logits = np.array([4.0, 2.0, 0.5, -1.0])
class_names = ['cat', 'dog', 'bird', 'fish']
temperatures = [0.5, 1.0, 2.0, 5.0]

# TODO: For each temperature:
for tau in temperatures:
    # a) Compute softmax distribution
    p = None  # softmax(logits, tau)
    
    # b) Compute entropy
    H = None  # -sum(p * log(p))
    
    # c) Compute effective classes
    eff = None  # exp(H)
    
    # d) Print results
    pass

# TODO: Analyse dark knowledge
# What ratios between non-top classes are preserved?

In [ ]:
# === Exercise 5: Solution ===

logits = np.array([4.0, 2.0, 0.5, -1.0])
class_names = ['cat', 'dog', 'bird', 'fish']
temperatures = [0.5, 1.0, 2.0, 5.0]
max_H = np.log(len(logits))  # Maximum entropy (uniform)

print('=== Temperature Effect on Softmax Distribution ===')
print(f'Logits: {list(logits)}')
print(f'Classes: {class_names}\n')

rows = []
distributions = {}
for tau in temperatures:
    p = softmax(logits, tau)
    H = entropy(p)
    eff_classes = np.exp(H)
    distributions[tau] = p
    
    rows.append([f'{tau:.1f}',
                 f'{p[0]:.4f}', f'{p[1]:.4f}',
                 f'{p[2]:.4f}', f'{p[3]:.4f}',
                 f'{H:.4f}',
                 f'{eff_classes:.2f}'])

print_table(['τ', 'P(cat)', 'P(dog)', 'P(bird)', 'P(fish)',
             'Entropy', 'Eff Classes'], rows, col_width=10)

# Dark knowledge analysis
print('\n=== Dark Knowledge Analysis ===')
print('Relative probabilities of non-cat classes reveal similarity structure:\n')

for tau in temperatures:
    p = distributions[tau]
    print(f'τ = {tau}:')
    print(f'  P(dog)/P(bird) = {p[1]/p[2]:.3f}')
    print(f'  P(dog)/P(fish) = {p[1]/p[3]:.3f}')
    print(f'  P(bird)/P(fish) = {p[2]/p[3]:.3f}')
    print(f'  Top-1 confidence: {p[0]*100:.1f}%\n')

print('Key Insights:')
print('  1. At τ=0.5: over-confident; nearly 100% on cat; other probs invisible')
print('  2. At τ=1.0: confident but some dark knowledge visible')
print('  3. At τ=2.0: softer; ratios between non-top classes more apparent')
print('  4. At τ=5.0: nearly uniform but PRESERVES similarity structure')
print('  5. P(dog)/P(bird) ratio preserved at all τ — this IS the dark knowledge')
print('  6. Dog is more similar to cat than bird is — student learns this from soft labels')

# Entropy as fraction of maximum
print(f'\nEntropy as % of maximum ({max_H:.4f}):')
for tau in temperatures:
    p = distributions[tau]
    H = entropy(p)
    print(f'  τ={tau:.1f}: {H/max_H*100:.1f}% of max entropy')

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    x_pos = np.arange(len(class_names))
    width = 0.18
    for i, tau in enumerate(temperatures):
        p = distributions[tau]
        axes[0].bar(x_pos + i*width, p, width, label=f'τ={tau}')
    axes[0].set_xticks(x_pos + 1.5*width)
    axes[0].set_xticklabels(class_names)
    axes[0].set_ylabel('Probability')
    axes[0].set_title('Distributions at Different Temperatures')
    axes[0].legend()
    
    taus_fine = np.linspace(0.1, 10, 200)
    ents = [entropy(softmax(logits, t)) for t in taus_fine]
    axes[1].plot(taus_fine, ents, 'b-')
    axes[1].axhline(max_H, color='r', linestyle='--', label='Max entropy')
    axes[1].set_xlabel('Temperature τ')
    axes[1].set_ylabel('Entropy (nats)')
    axes[1].set_title('Entropy vs Temperature')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('temperature_effect.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved temperature_effect.png')

## Exercise 6: QLoRA Memory Calculation

LLaMA-3 8B model. Compare memory for:

**A) Full BF16 fine-tuning:**
- Weights: 8B × 2 bytes (BF16)
- Gradients: 8B × 2 bytes
- Adam states: 8B × 2 × 4 bytes (two FP32 moments)

**B) QLoRA (NF4 base + LoRA r=16 on Q,V in 32 layers):**
- Base weights: 8B × 0.5 bytes (NF4) + quantization overhead
- LoRA: 2 × 32 × (d_model × r + r × d_model) × 2 bytes (BF16)
- LoRA gradients + Adam states for adapters only

In [ ]:
# === Exercise 6: Scaffold ===

# LLaMA-3 8B architecture
n_params = 8e9        # Total parameters
d_model = 4096        # Hidden dimension
n_layers = 32         # Transformer layers
n_heads = 32          # Attention heads
lora_r = 16           # LoRA rank
lora_targets = 2      # Q and V projections

# TODO: BF16 full fine-tuning memory
bf16_weights = None        # n_params * 2 bytes
bf16_gradients = None      # n_params * 2 bytes
bf16_adam = None            # n_params * 2 * 4 bytes (two FP32 moments)
bf16_total = None

# TODO: QLoRA memory
qlora_base_weights = None   # n_params * 0.5 bytes (NF4)
qlora_quant_overhead = None # Quantization constants
qlora_lora_params = None    # Number of LoRA parameters
qlora_lora_weights = None   # LoRA weights in BF16
qlora_lora_grads = None     # LoRA gradients
qlora_lora_adam = None      # Adam states for LoRA only
qlora_total = None

# TODO: Compute savings

In [ ]:
# === Exercise 6: Solution ===

# LLaMA-3 8B architecture
n_params = 8e9
d_model = 4096
n_layers = 32
n_heads = 32
lora_r = 16
lora_targets = 2  # Q and V

print('=== QLoRA vs BF16 Fine-Tuning Memory ===')
print(f'Model: LLaMA-3 8B (d={d_model}, L={n_layers})\n')

# --- A) BF16 Full Fine-Tuning ---
bf16_weights = n_params * 2                  # BF16 weights
bf16_gradients = n_params * 2                # BF16 gradients
bf16_adam_m = n_params * 4                   # FP32 first moment
bf16_adam_v = n_params * 4                   # FP32 second moment
bf16_total = bf16_weights + bf16_gradients + bf16_adam_m + bf16_adam_v

print('A) BF16 Full Fine-Tuning:')
print(f'  Weights:    {fmt_bytes(bf16_weights)}')
print(f'  Gradients:  {fmt_bytes(bf16_gradients)}')
print(f'  Adam m:     {fmt_bytes(bf16_adam_m)}')
print(f'  Adam v:     {fmt_bytes(bf16_adam_v)}')
print(f'  TOTAL:      {fmt_bytes(bf16_total)}')

# --- B) QLoRA ---
# Base weights in NF4 (4-bit = 0.5 bytes)
qlora_base = n_params * 0.5

# Quantization overhead: one FP32 scale per group of 64
n_groups = n_params / 64
quant_scales = n_groups * 4  # FP32 scales
# Double quantization: quantize FP32 scales to FP8
double_quant_scales = n_groups * 1  # FP8 instead of FP32
quant_overhead = double_quant_scales  # With double quantization

# LoRA parameters: for Q and V projections in each layer
# Each LoRA: A (d_model × r) + B (r × d_model)
lora_params_per_target = d_model * lora_r + lora_r * d_model  # A + B
total_lora_params = lora_targets * n_layers * lora_params_per_target

qlora_lora_weights = total_lora_params * 2     # BF16
qlora_lora_grads = total_lora_params * 2       # BF16 gradients
qlora_lora_adam_m = total_lora_params * 4       # FP32 first moment
qlora_lora_adam_v = total_lora_params * 4       # FP32 second moment

qlora_total = (qlora_base + quant_overhead +
               qlora_lora_weights + qlora_lora_grads +
               qlora_lora_adam_m + qlora_lora_adam_v)

print(f'\nB) QLoRA (NF4 + LoRA r={lora_r} on Q,V):')
print(f'  Base NF4:        {fmt_bytes(qlora_base)}')
print(f'  Quant overhead:  {fmt_bytes(quant_overhead)} (double quantized)')
print(f'  LoRA params:     {total_lora_params:,.0f} ({total_lora_params/n_params*100:.3f}% of base)')
print(f'  LoRA weights:    {fmt_bytes(qlora_lora_weights)}')
print(f'  LoRA gradients:  {fmt_bytes(qlora_lora_grads)}')
print(f'  LoRA Adam m:     {fmt_bytes(qlora_lora_adam_m)}')
print(f'  LoRA Adam v:     {fmt_bytes(qlora_lora_adam_v)}')
print(f'  TOTAL:           {fmt_bytes(qlora_total)}')

# --- Comparison ---
print(f'\n=== Savings ===')
print(f'  BF16 total:  {fmt_bytes(bf16_total)}')
print(f'  QLoRA total: {fmt_bytes(qlora_total)}')
print(f'  Reduction:   {bf16_total/qlora_total:.1f}× less memory')
print(f'  Saved:       {fmt_bytes(bf16_total - qlora_total)}')
print(f'\n  BF16: needs ~{bf16_total/80e9:.0f}× A100-80GB GPUs')
print(f'  QLoRA: fits on 1× A100-80GB (or even smaller GPUs)')

# Breakdown table
print('\n=== Memory Breakdown ===')
rows = [
    ['Weights', fmt_bytes(bf16_weights), fmt_bytes(qlora_base + quant_overhead)],
    ['Gradients', fmt_bytes(bf16_gradients), fmt_bytes(qlora_lora_grads)],
    ['Adam states', fmt_bytes(bf16_adam_m + bf16_adam_v),
     fmt_bytes(qlora_lora_adam_m + qlora_lora_adam_v)],
    ['TOTAL', fmt_bytes(bf16_total), fmt_bytes(qlora_total)],
]
print_table(['Component', 'BF16', 'QLoRA'], rows, col_width=18)

## Exercise 7: Sensitivity Analysis for Mixed-Precision

A 12-layer model. Per-layer PPL increase when quantized to INT4:

| Layer | PPL Δ |
|-------|-------|
| 1 | +3.2 |
| 2 | +1.8 |
| 3 | +1.0 |
| 4 | +0.6 |
| 5 | +0.4 |
| 6 | +0.4 |
| 7 | +0.5 |
| 8 | +0.6 |
| 9 | +0.8 |
| 10 | +1.2 |
| 11 | +1.5 |
| 12 | +1.8 |

Design a mixed-precision scheme that achieves average ≤ 5 bits per weight while minimising total PPL increase.

In [ ]:
# === Exercise 7: Scaffold ===

# Per-layer PPL increase at INT4
ppl_delta_int4 = {
    1: 3.2, 2: 1.8, 3: 1.0, 4: 0.6, 5: 0.4, 6: 0.4,
    7: 0.5, 8: 0.6, 9: 0.8, 10: 1.2, 11: 1.5, 12: 1.8
}

# Available bit widths and their relative PPL impact
# Assume: INT8 ≈ 0.1× the INT4 PPL delta; INT6 ≈ 0.3× INT4 delta
bit_options = [4, 5, 6, 8]

# TODO: Design mixed-precision allocation
# Constraint: average bits ≤ 5
# Objective: minimise total PPL increase

# TODO: Assign each layer a bit width
layer_bits = {}  # layer -> bits

# TODO: Compute average bits and estimated total PPL delta

In [ ]:
# === Exercise 7: Solution ===

ppl_delta_int4 = {
    1: 3.2, 2: 1.8, 3: 1.0, 4: 0.6, 5: 0.4, 6: 0.4,
    7: 0.5, 8: 0.6, 9: 0.8, 10: 1.2, 11: 1.5, 12: 1.8
}

# PPL reduction factors relative to INT4
# Each extra bit ≈ halves quantization error; PPL impact scales ~linearly with error
ppl_factor = {
    4: 1.0,    # Full INT4 impact
    5: 0.50,   # ~50% of INT4 impact
    6: 0.25,   # ~25% of INT4 impact
    8: 0.05,   # ~5% of INT4 impact (near-lossless)
}

n_layers = 12
target_avg_bits = 5.0
total_bit_budget = target_avg_bits * n_layers  # 60 bits total

print('=== Sensitivity Analysis ===')
print(f'Target: average ≤ {target_avg_bits} bits ({total_bit_budget:.0f} total)\n')

# Strategy: sort layers by sensitivity (PPL delta); assign more bits to sensitive layers
sorted_layers = sorted(ppl_delta_int4.items(), key=lambda x: x[1], reverse=True)

print('Layers sorted by sensitivity:')
for layer, delta in sorted_layers:
    print(f'  Layer {layer:2d}: PPL Δ = {delta:.1f}')

# Greedy allocation: start all at 4-bit; upgrade most sensitive layers first
layer_bits = {l: 4 for l in range(1, 13)}
remaining_budget = total_bit_budget - sum(layer_bits.values())  # 60 - 48 = 12 extra bits

print(f'\nGreedy allocation: {remaining_budget:.0f} extra bits to distribute')

# Upgrade most sensitive layers
for layer, delta in sorted_layers:
    if remaining_budget <= 0:
        break
    # How many extra bits to give this layer?
    upgrade = min(4, remaining_budget)  # Max upgrade to 8-bit
    layer_bits[layer] = 4 + int(upgrade)
    remaining_budget -= upgrade

# Compute results
avg_bits = sum(layer_bits.values()) / n_layers
total_ppl_mixed = sum(
    ppl_delta_int4[l] * ppl_factor.get(layer_bits[l], 0.5)
    for l in range(1, 13)
)
total_ppl_uniform_4bit = sum(ppl_delta_int4.values())
total_ppl_uniform_8bit = sum(v * ppl_factor[8] for v in ppl_delta_int4.values())

print(f'\n=== Mixed-Precision Allocation ===')
rows = []
for l in range(1, 13):
    bits = layer_bits[l]
    ppl_est = ppl_delta_int4[l] * ppl_factor.get(bits, 0.5)
    rows.append([f'Layer {l}', f'{ppl_delta_int4[l]:.1f}',
                 f'{bits}', f'{ppl_est:.3f}'])
print_table(['Layer', 'INT4 PPL Δ', 'Bits', 'Est PPL Δ'], rows, col_width=12)

print(f'\n=== Summary ===')
print(f'Average bits:        {avg_bits:.2f} (target ≤ {target_avg_bits})')
print(f'Mixed-precision PPL: {total_ppl_mixed:.3f}')
print(f'Uniform INT4 PPL:    {total_ppl_uniform_4bit:.3f}')
print(f'Uniform INT8 PPL:    {total_ppl_uniform_8bit:.3f}')
print(f'Improvement vs INT4: {total_ppl_uniform_4bit/total_ppl_mixed:.1f}× less PPL degradation')
print(f'\nKey insight: most sensitive layers (1, 2, 12) get highest precision.')
print(f'Insensitive middle layers (5, 6, 7) stay at INT4 with minimal PPL cost.')

## Exercise 8: Distillation Capacity Gap

Teacher: 70B parameters. Target student: 1B parameters.

1. Estimate quality loss for direct 70B → 1B distillation (70× compression)
2. Design teacher assistant cascade: choose intermediate sizes
3. Estimate quality retention at each stage
4. Compare direct vs cascade distillation

In [ ]:
# === Exercise 8: Scaffold ===

teacher_params = 70e9  # 70B
student_params = 1e9   # 1B

# TODO: Step 1 — Estimate compression ratio and quality loss
compression_ratio = None  # teacher / student
# Rule of thumb: quality ~ log(params), so quality_ratio ≈ log(student)/log(teacher)

# TODO: Step 2 — Design cascade
# Choose 2-3 intermediate sizes
cascade = None  # [70B, ???, ???, 1B]

# TODO: Step 3 — Estimate quality at each stage
# Distillation typically retains 90-95% of teacher quality
# when compression ratio < 5×

# TODO: Step 4 — Compare

In [ ]:
# === Exercise 8: Solution ===

teacher_params = 70e9
student_params = 1e9

print('=== Distillation Capacity Gap Analysis ===')
print(f'Teacher: {teacher_params/1e9:.0f}B, Student: {student_params/1e9:.0f}B\n')

# Step 1: Compression analysis
compression = teacher_params / student_params
print(f'Step 1: Direct compression ratio = {compression:.0f}×')

# Quality estimation model:
# Based on scaling laws, quality ~ C * log(N) for model size N
# Distillation efficiency depends on capacity gap
def distillation_efficiency(teacher_size, student_size):
    """Estimate fraction of teacher knowledge retained.
    Based on empirical observations from distillation papers."""
    ratio = teacher_size / student_size
    if ratio <= 2:
        return 0.95   # Small gap: 95% retention
    elif ratio <= 5:
        return 0.90   # Moderate gap: 90%
    elif ratio <= 10:
        return 0.82   # Significant gap: 82%
    elif ratio <= 20:
        return 0.72   # Large gap: 72%
    elif ratio <= 50:
        return 0.60   # Very large gap
    else:
        return 0.50   # Extreme gap: 50%

# Direct distillation
direct_efficiency = distillation_efficiency(teacher_params, student_params)
print(f'Direct 70B→1B efficiency: {direct_efficiency:.0%}')
print(f'  → Student retains ~{direct_efficiency:.0%} of teacher quality')
print(f'  → This is suboptimal due to large capacity gap\n')

# Step 2: Design cascade
# Rule: each step should have compression ≤ 5× for good knowledge transfer
cascades = {
    'Direct': [70, 1],
    '2-step': [70, 7, 1],
    '3-step': [70, 13, 3, 1],
    '4-step': [70, 30, 7, 3, 1],
}

print('Step 2: Cascade designs')
for name, sizes in cascades.items():
    ratios = [sizes[i]/sizes[i+1] for i in range(len(sizes)-1)]
    print(f'  {name}: {" → ".join(f"{s}B" for s in sizes)}')
    print(f'    Ratios: {" → ".join(f"{r:.1f}×" for r in ratios)}')

# Step 3: Estimate quality at each stage
print('\nStep 3: Quality estimation per cascade')
print('(Teacher quality = 100.0)\n')

results = {}
for name, sizes in cascades.items():
    quality = 100.0  # Teacher quality
    stages = []
    for i in range(len(sizes)-1):
        t_size = sizes[i] * 1e9
        s_size = sizes[i+1] * 1e9
        eff = distillation_efficiency(t_size, s_size)
        quality *= eff
        stages.append((sizes[i], sizes[i+1], eff, quality))
    results[name] = (quality, stages)
    
    print(f'{name}:')
    for t, s, eff, q in stages:
        print(f'  {t}B → {s}B: efficiency={eff:.0%}, quality={q:.1f}')
    print(f'  Final quality: {quality:.1f}\n')

# Step 4: Comparison
print('=== Step 4: Comparison ===')
rows = []
for name, (quality, stages) in results.items():
    n_steps = len(stages)
    total_cost = sum(s[0] + s[1] for s in stages)  # Rough training cost
    rows.append([name, f'{n_steps}', f'{quality:.1f}',
                 f'{quality/100*100:.1f}%', f'{total_cost}B'])
print_table(['Cascade', 'Steps', 'Quality', 'Retention', 'Train Cost'], rows, col_width=14)

print('\nRecommendation:')
best_name = max(results.keys(), key=lambda k: results[k][0])
print(f'  Best cascade: {best_name}')
print(f'  Quality: {results[best_name][0]:.1f} vs Direct: {results["Direct"][0]:.1f}')
print(f'  Improvement: +{results[best_name][0]-results["Direct"][0]:.1f} quality points')
print(f'\nKey insight: teacher assistant bridges the capacity gap.')
print(f'  Multiple small steps > one large step for knowledge transfer.')
print(f'  Each step stays within the "efficient distillation zone" (ratio ≤ 5×).')

print()
print('=' * 60)
print('   ALL EXERCISES COMPLETE')
print('=' * 60)
print()
print('Summary of exercises:')
print('  1. Uniform quantization: s = absmax / q_max; MSE ≈ s²/12')
print('  2. NF4 vs INT4: NF4 wins for Gaussian data (quantile-optimal)')
print('  3. GPTQ compensation: Hessian-guided updates reduce output error')
print('  4. Distillation loss: KL(T||S) × τ² + α · CE(hard)')
print('  5. Temperature: soft labels reveal dark knowledge at high τ')
print('  6. QLoRA memory: ~10× less memory than BF16 fine-tuning')
print('  7. Sensitivity: mixed-precision > uniform at same avg bits')
print('  8. Capacity gap: cascade distillation > direct for large gaps')